# Dataset exploration

Explore the multilingual natural-language inference data before choosing a validation strategy or model. The notebook reuses the project's loading and schema-validation helpers and leaves observations for us to record as we work.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from watson_dataset_config import LABEL_NAMES
from watson_dataset_helpers import (
    build_dataset_overview,
    load_datasets,
    validate_dataset_columns,
)

pd.set_option("display.max_colwidth", 180)
plt.style.use("seaborn-v0_8-whitegrid")

## Load and validate the data

In [ ]:
datasets = load_datasets()
validate_dataset_columns(datasets)

train = datasets["train"].copy()
test = datasets["test"].copy()
submission = datasets["sample_submission"].copy()

build_dataset_overview(datasets)

In [ ]:
display(train.head())
display(test.head())
display(submission.head())

### Initial observations

- Dataset shape and schema:
- Missing or unexpected values:
- Questions to revisit:

## Class balance

In [ ]:
label_counts = (
    train["label"]
    .value_counts()
    .sort_index()
    .rename(index=LABEL_NAMES)
    .rename("count")
    .to_frame()
)
label_counts["proportion"] = label_counts["count"] / len(train)
label_counts

In [ ]:
labels_by_language = pd.crosstab(
    train["language"],
    train["label"].map(LABEL_NAMES),
)
label_proportions_by_language = labels_by_language.div(
    labels_by_language.sum(axis=1), axis=0
)

display(labels_by_language)
display(label_proportions_by_language.round(3))

## Language coverage

In [ ]:
language_counts = pd.concat(
    [
        train["language"].value_counts().rename("train"),
        test["language"].value_counts().rename("test"),
    ],
    axis=1,
).fillna(0).astype(int)
language_counts["train_proportion"] = language_counts["train"] / len(train)
language_counts["test_proportion"] = language_counts["test"] / len(test)
language_counts.sort_values("train", ascending=False)

In [ ]:
language_counts[["train", "test"]].sort_values("train").plot.barh(
    figsize=(9, 7),
    title="Language representation in train and test",
)
plt.xlabel("Examples")
plt.ylabel("Language")
plt.tight_layout()

## Text lengths

Character counts are a first-pass description. They are not token counts, and their meaning varies across writing systems.

In [ ]:
TEXT_COLUMNS = ["premise", "hypothesis"]

for frame in (train, test):
    for column in TEXT_COLUMNS:
        frame[f"{column}_chars"] = frame[column].str.len()
        frame[f"{column}_words"] = frame[column].str.split().str.len()

length_columns = [
    "premise_chars",
    "hypothesis_chars",
    "premise_words",
    "hypothesis_words",
]
train[length_columns].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for column, axis in zip(["premise_chars", "hypothesis_chars"], axes):
    upper_bound = train[column].quantile(0.99)
    train.loc[train[column] <= upper_bound, column].plot.hist(
        bins=40, ax=axis, title=f"{column} (through 99th percentile)"
    )
    axis.set_xlabel("Characters")
plt.tight_layout()

In [ ]:
lengths_by_language = train.groupby("language")[length_columns].agg(
    ["min", "median", "mean", "max"]
).round(1)
lengths_by_language

## Very short text audit

Use fewer than 30 characters as an adjustable screening threshold. Python's `str.len()` counts Unicode code points; a short count does not necessarily mean little semantic content, especially across writing systems.

In [ ]:
SHORT_TEXT_THRESHOLD = 30

short_premise = train["premise_chars"] < SHORT_TEXT_THRESHOLD
short_hypothesis = train["hypothesis_chars"] < SHORT_TEXT_THRESHOLD

short_text_summary = pd.Series(
    {
        "short premise": short_premise.sum(),
        "short hypothesis": short_hypothesis.sum(),
        "either is short": (short_premise | short_hypothesis).sum(),
        "both are short": (short_premise & short_hypothesis).sum(),
    },
    name="examples",
).to_frame()
short_text_summary["proportion_of_train"] = (
    short_text_summary["examples"] / len(train)
)
short_text_summary

In [ ]:
short_counts_by_language = (
    train.assign(
        short_premise=short_premise,
        short_hypothesis=short_hypothesis,
        either_short=short_premise | short_hypothesis,
        both_short=short_premise & short_hypothesis,
    )
    .groupby("language")
    .agg(
        examples=("id", "size"),
        short_premises=("short_premise", "sum"),
        short_hypotheses=("short_hypothesis", "sum"),
        either_short=("either_short", "sum"),
        both_short=("both_short", "sum"),
    )
)
short_counts_by_language["either_short_rate"] = (
    short_counts_by_language["either_short"]
    / short_counts_by_language["examples"]
)
short_counts_by_language.sort_values("either_short_rate", ascending=False)

In [ ]:
short_text_candidates = (
    train.loc[
        short_premise | short_hypothesis,
        [
            "id",
            "language",
            "label",
            "premise_chars",
            "hypothesis_chars",
            "premise",
            "hypothesis",
        ],
    ]
    .assign(label_name=lambda frame: frame["label"].map(LABEL_NAMES))
    .sort_values(["premise_chars", "hypothesis_chars"])
)
short_text_candidates

Change `SHORT_TEXT_THRESHOLD` above to widen or narrow the audit. The candidate table remains a DataFrame, so it can also be filtered by language, label, or either length column before displaying selected rows.

## Duplicate and conflicting examples

In [ ]:
pair_columns = ["premise", "hypothesis"]
duplicate_pair_mask = train.duplicated(pair_columns, keep=False)
duplicate_pairs = train.loc[duplicate_pair_mask].sort_values(pair_columns)

conflicting_pairs = (
    train.groupby(pair_columns, dropna=False)["label"]
    .nunique()
    .loc[lambda counts: counts > 1]
)

pd.Series(
    {
        "rows belonging to duplicated pairs": len(duplicate_pairs),
        "unique duplicated pairs": train.loc[duplicate_pair_mask, pair_columns].drop_duplicates().shape[0],
        "pairs with conflicting labels": len(conflicting_pairs),
    }
)

In [ ]:
duplicate_pairs[["id", "language", "label", *pair_columns]]

## Train/test text overlap

Exact overlap can affect how we interpret validation and test performance. This is a diagnostic only; test labels remain unknown.

In [ ]:
train_pairs = pd.MultiIndex.from_frame(train[pair_columns])
test_pairs = pd.MultiIndex.from_frame(test[pair_columns])

overlap_summary = pd.Series(
    {
        "test rows with exact train pair": test_pairs.isin(train_pairs).sum(),
        "test rows with premise seen in train": test["premise"].isin(train["premise"]).sum(),
        "test rows with hypothesis seen in train": test["hypothesis"].isin(train["hypothesis"]).sum(),
    }
)
overlap_summary

In [ ]:
test.loc[
    test_pairs.isin(train_pairs),
    ["id", "language", "premise", "hypothesis"],
]

## Findings and validation implications

- Class balance:
- Language imbalance and train/test differences:
- Notable length patterns:
- Very short examples worth inspecting:
- Duplicate or conflicting pairs:
- Train/test overlap:
- Implications for a leakage-safe, language-aware split: